In [25]:
import os
import random
import subprocess
import asyncio
import edge_tts
import cv2

from IPython.display import Video

DATASET_PATH = "dataset/viseme_dataset_aligned"
ALIGNED_DATASET_PATH = "dataset/viseme_dataset_aligned"

TEMP_VIDEO = "temp_concat.mp4"
FINAL_VIDEO = "final.mp4"
AUDIO_FILE = "response.mp3"

In [26]:
VISeme_MAP = {
    "a": "viseme_open",
    "e": "viseme_wide",
    "i": "viseme_wide",
    "o": "viseme_round",
    "u": "viseme_round",
    "b": "viseme_full_closed",
    "p": "viseme_full_closed",
    "m": "viseme_full_closed",
    "f": "viseme_partial_closed",
    "v": "viseme_partial_closed",
    "s": "viseme_sibilant",
    "z": "viseme_sibilant",
    "t": "viseme_tongue",
    "d": "viseme_tongue",
    "n": "viseme_tongue",
    "l": "viseme_tongue",
    "r": "viseme_neutral",
    "k": "viseme_neutral",
    "g": "viseme_neutral",
    "h": "viseme_neutral",
    "j": "viseme_neutral",
    "q": "viseme_neutral",
    "w": "viseme_round",
    "x": "viseme_sibilant",
    "y": "viseme_wide",
}

In [27]:
def text_to_visemes(text):
    words = text.lower().split()
    return [VISeme_MAP.get(word[0], "viseme_neutral") for word in words]

In [28]:
print(text_to_visemes("Hi good morning"))

['viseme_neutral', 'viseme_neutral', 'viseme_full_closed']


In [29]:
def select_clip(viseme):
    folder_path = os.path.join(DATASET_PATH, viseme)
    files = [f for f in os.listdir(folder_path) if f.endswith(".mp4")]
    return os.path.join(folder_path, random.choice(files))


def create_concat_file(clip_paths, filename="list.txt"):
    with open(filename, "w") as f:
        for path in clip_paths:
            f.write(f"file '{os.path.abspath(path)}'\n")
    return filename


def concatenate_clips(list_file, output_video):
    command = [
        "ffmpeg", "-y",
        "-f", "concat",
        "-safe", "0",
        "-i", list_file,
        "-c", "copy",
        output_video
    ]
    subprocess.run(command)


def attach_audio(video_path, audio_path, final_output):
    command = [
        "ffmpeg", "-y",
        "-i", video_path,
        "-i", audio_path,
        "-map", "0:v:0",
        "-map", "1:a:0",
        "-c:v", "copy",
        "-c:a", "aac",
        "-shortest",
        final_output
    ]
    subprocess.run(command)

In [30]:
def generate_video(text, audio_path, output_path):
    viseme_sequence = text_to_visemes(text)

    clip_paths = [select_clip(v) for v in viseme_sequence]

    list_file = create_concat_file(clip_paths)

    concatenate_clips(list_file, TEMP_VIDEO)
    attach_audio(TEMP_VIDEO, audio_path, output_path)

    os.remove(list_file) 
    os.remove(TEMP_VIDEO)
    print("Video generated:", output_path)

In [31]:
async def generate_audio_async(text: str, output_path: str):
    communicate = edge_tts.Communicate(
        text=text,
        voice="en-IN-PrabhatNeural"
    )
    await communicate.save(output_path)

In [32]:
text = "Hi good morning how are you"

# 1️⃣ Generate audio
await generate_audio_async(text, AUDIO_FILE)

# 2️⃣ Generate video
generate_video(text, AUDIO_FILE, FINAL_VIDEO)

# 3️⃣ Display video
Video(FINAL_VIDEO, embed=True)

Video generated: final.mp4


ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [33]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

In [34]:
def get_face_center(video_path):
    cap = cv2.VideoCapture(video_path)
    ret, frame = cap.read()
    cap.release()

    if not ret:
        return None

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    if len(faces) == 0:
        return None

    x, y, w, h = faces[0]
    cx = x + w // 2
    cy = y + h // 2

    return cx, cy, frame.shape[1], frame.shape[0]

In [35]:
def center_face(video_path, output_path):
    result = get_face_center(video_path)
    if result is None:
        print("No face detected:", video_path)
        return

    cx, cy, width, height = result

    dx = (width // 2) - cx
    dy = (height // 2) - cy

    subprocess.run([
        "ffmpeg", "-y",
        "-i", video_path,
        "-vf", f"pad={width}:{height}:{dx}:{dy}:black",
        "-c:a", "copy",
        output_path
    ])

In [36]:
os.makedirs(ALIGNED_DATASET_PATH, exist_ok=True)

for folder in os.listdir(DATASET_PATH):
    src_folder = os.path.join(DATASET_PATH, folder)
    dst_folder = os.path.join(ALIGNED_DATASET_PATH, folder)

    os.makedirs(dst_folder, exist_ok=True)

    for file in os.listdir(src_folder):
        if file.endswith(".mp4"):
            input_path = os.path.join(src_folder, file)
            output_path = os.path.join(dst_folder, file)

            center_face(input_path, output_path)

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [37]:
!ffmpeg -i response.mp3
!ffmpeg -i temp_concat.mp4
!ffmpeg -i final.mp4

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena